In [ ]:

%%capture
!pip install -q openenv unsloth transformers trl datasets wandb \
    google-api-python-client google-auth-httplib2 \
    google-auth-oauthlib matplotlib huggingface-hub

 Configuration (Fill in your credentials)

In [ ]:
HF_TOKEN        = ""
CURSOR_API_KEY  = ""
BASE_MODEL      = "unsloth/Qwen2.5-7B-Instruct"
OUTPUT_MODEL    = "your-hf-username/ai-task-orchrestration"
MAX_STEPS       = 500
BATCH_SIZE      = 4
LR              = 2e-9
SAVE_STEPS      = 100
EVAL_STEPS      = 50


In [ ]:
import os
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN)
    os.environ["HF_TOKEN"] = HF_TOKEN

if CURSOR_API_KEY:
    os.environ["CURSOR_API_KEY"] = CURSOR_API_KEY

In [ ]:
import os
import sys

if not os.path.exists('env'):
    print("Alfred environment files not found.")
    print("Please upload the alfred-openenv folder to this Colab session.")
    print("Ensure the directory structure is correct before proceeding.")

sys.path.insert(0, '.')

try:
    from env.butler_env import ButlerEnvironment
    from env.observation import build_observation_prompt, SYSTEM_PROMPT
    from env.action_space import parse_llm_output, validate_action
    from agents.orchestrator import Orchestrator

    env = ButlerEnvironment()
    obs = env.reset()

    print(f"Queue Size: {len(obs['queue'])}")
    print(f"Focus Task: {obs['current_todo']['text']}")
    print(f"Tier Classification: {obs['current_todo']['tier']}")

    queue_order = [t['tier'] for t in obs['queue']]
    print(f"Protocol Order: {queue_order}")

except ModuleNotFoundError:
    print("Error: Alfred environment modules missing. Check file uploads.")

Generate Synthetic Dataset


In [ ]:
from data.synthetic_todos import save_dataset, generate_batch
from datasets import Dataset
import json
import os

os.makedirs("data", exist_ok=True)

save_dataset(
    path="data/alfred_dataset.json",
    n_train=500,
    n_eval=100,
    n_test=100
)

with open("data/alfred_dataset.json") as f:
    raw = json.load(f)

train_data = [{"episode": ep} for ep in raw["train"]]
eval_data = [{"episode": ep} for ep in raw["eval"]]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print(f"Training Set: {len(train_dataset)} episodes")
print(f"Evaluation Set: {len(eval_dataset)} episodes")

Load Model with Unsloth


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"Alfred Intelligence Loaded: {BASE_MODEL}")
model.print_trainable_parameters()

In [ ]:
from env.alfred_env import AlfredEnvironment
from env.action_space import parse_llm_output, validate_action

reward_env = AlfredEnvironment()

def alfred_reward_fn(completions, prompts, **kwargs):
    rewards = []
    for completion in completions:
        try:
            action = parse_llm_output(completion)
            if action is None:
                rewards.append(0.0)
                continue

            valid, error = validate_action(action)
            if not valid:
                rewards.append(0.1)
                continue

            reward_env.reset()
            _, reward, _, info = reward_env.step(action)
            rewards.append(float(reward))

        except Exception as e:
            rewards.append(0.0)

    return rewards

print("Alfred Reward Function Defined.")

format dataset for GRPO


In [ ]:
from env.observation import build_observation_prompt, SYSTEM_PROMPT
from agents.orchestrator import Orchestrator

orch = Orchestrator()

def format_episode(example):
    queue = orch.sort_queue(example["episode"])
    obs = {
        "queue": queue,
        "current_todo": queue[0] if queue else None,
        "user_context": {
            "name": "Master",
            "timezone": "Asia/Kolkata",
            "communication_style": "formal",
            "role": "Engineer",
        },
        "step": 0,
        "max_steps": 10,
    }
    return {"prompt": f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{build_observation_prompt(obs)}\n<|assistant|>\n"}

train_dataset = train_dataset.map(format_episode, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(format_episode, remove_columns=eval_dataset.column_names)

print(f"Dataset Formatted. Training: {len(train_dataset)}, Evaluation: {len(eval_dataset)}")
print(f"Sample: {train_dataset[0]['prompt'][:500]}")

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="./alfred-grpo-output",
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    max_steps=MAX_STEPS,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    report_to="wandb",
    run_name="alfred-grpo-run-1",
    max_new_tokens=256,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=alfred_reward_fn,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete.")

plotting of reward curves



In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("assets", exist_ok=True)

log_history = trainer.state.log_history

steps   = [x["step"]   for x in log_history if "reward" in x]
rewards = [x["reward"] for x in log_history if "reward" in x]

plt.figure(figsize=(10, 5))
plt.plot(steps, rewards, color="#6C63FF", linewidth=2, label="Butler GRPO")
plt.axhline(
    y=0.21, color="#FF6B6B", linestyle="--",
    linewidth=1.5, label="Random baseline (0.21)"
)
plt.xlabel("Training Step")
plt.ylabel("Reward (0–1)")
plt.title("Butler GRPO Training — Reward vs Step")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("assets/reward_curve.png", dpi=150)
plt.show()
print("Saved: assets/reward_curve.png")
